# 🛠️ Giorno 2 - Lab 1: MCP (Model Context Protocol) & Tool Calling con Ollama

Benvenuti al primo laboratorio del **Giorno 2**!
In questo notebook metteremo in pratica i concetti presentati durante la mattina di teoria:
1. **Il problema N×M** e la necessità di una standardizzazione della comunicazione (Protocollo vs Framework vs Architettura).
2. **Architettura MCP**: Host, Client, Server e le sue 3 primitive (**Tools**, **Resources**, **Prompts**).
3. **Anatomia di una Tool Call**: Richiesta e Risposta strutturata JSON.
4. **Tool Calling reale** collegando il modello locale **`llama3.2`** eseguito tramite **Ollama** al nostro **Server MCP** in esecuzione su porta 8000.

---
### 🎯 Avvio del Server MCP
Per eseguire questo notebook, assicurati di avere il server MCP in esecuzione sul tuo terminale:
```bash
python simple_mcp_server.py
```
*(Se il server non è in esecuzione in background, il notebook include anche il fallback in memoria!)*

## 1. Setup dell'Ambiente e Connessione a Ollama & Server MCP
Verifichiamo la connettività verso **Ollama** (`http://localhost:11434`) e verso il nostro **Server MCP** (`http://localhost:9000`).

In [8]:
import requests
import json
from pprint import pprint
from mcp_client import MCPClient

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL_NAME = "llama3.2"

# 1. Verifica Ollama
try:
    resp = requests.get("http://localhost:11434/api/tags", timeout=2)
    if resp.status_code == 200:
        models = [m["name"] for m in resp.json().get("models", [])]
        print(f"✅ Connessione a Ollama riuscita! Modelli disponibili: {models}")
except Exception as e:
    print(f"⚠️ Ollama non raggiungibile ({e}). Assicurati di aver avviato 'ollama serve'.")

# 2. Connessione al Server MCP
mcp_client = MCPClient("http://127.0.0.1:9090")

if mcp_client.is_alive():
    print("✅ Server MCP connesso e in ascolto su http://127.0.0.1:9090/mcp")
else:
    print("⚠️ Server HTTP non rilevato: usiamo l'istanza mcp locale importata da simple_mcp_server.")
    from simple_mcp_server import mcp as local_mcp

✅ Connessione a Ollama riuscita! Modelli disponibili: ['qwen2.5:latest', 'erwan2/DeepSeek-R1-Distill-Qwen-7B:latest', 'nomic-embed-text:latest', 'qwen3.5:9b', 'qwen2.5-coder:7b', 'llama3.2:3b-instruct-q2_K', 'llama3.2:3b-instruct-q4_0', 'llama3.2:3b-instruct-q8_0', 'llama3.2:latest']
✅ Server MCP connesso e in ascolto su http://127.0.0.1:9090/mcp


## 2. Esplorazione delle 3 Primitive MCP
Interroghiamo il Server MCP per esplorare le tre capacità che espone:
1. **Resources**: La policy aziendale in sola lettura caricabile nel contesto.
2. **Prompts**: Il template standard di valutazione.
3. **Tools**: Le funzioni operative invocabili da Llama 3.2.

In [9]:
# Funzioni helper che usano il Server HTTP se attivo, oppure l'istanza locale
def get_mcp_resources():
    return mcp_client.list_resources() if mcp_client.is_alive() else [{"uri": "policy://acquisti-hardware"}]

def read_mcp_resource(uri):
    return mcp_client.read_resource(uri) if mcp_client.is_alive() else local_mcp.read_resource(uri)["contents"]

def get_mcp_tools():
    return mcp_client.list_tools() if mcp_client.is_alive() else local_mcp.get_tools_schema()

def execute_mcp_tool(name, args):
    if mcp_client.is_alive():
        return mcp_client.call_tool(name, args)
    return local_mcp.call_tool(name, args)

print("--- 📚 1. RISORSE ESPOSTE (Resources) ---")
resources = get_mcp_resources()
pprint(resources)

print("\n--- 📄 Lettura della risorsa Policy ---")
policy_content = read_mcp_resource("policy://acquisti-hardware")
print(policy_content)

--- 📚 1. RISORSE ESPOSTE (Resources) ---
[{'description': 'Restituisce il testo ufficiale della policy aziendale sugli '
                 'acquisti hardware.',
  'uri': 'policy://acquisti-hardware'}]

--- 📄 Lettura della risorsa Policy ---
# POLICY AZIENDALE ACQUISTI HARDWARE
- Limite max per Engineering: 1800€ (laptop Developer).
- Limite max per altri reparti: 1000€ (laptop Standard).
- Monitor e periferiche standard: max 300€ senza approvazione speciale.

REGOLE DI VALUTAZIONE:
1. SE Prezzo <= Limite_Reparto E Prezzo <= Budget_Residuo -> L'acquisto è APPROVATO AUTOMATICAMENTE (IN POLICY). Non chiamare 'request_manager_approval'.
2. SE Prezzo > Limite_Reparto OPPURE Prezzo > Budget_Residuo -> Serve approvazione: solo in questo caso chiama 'request_manager_approval'.



### Esplorazione dei Tools e dei loro Schemi JSON (JSON Schema)
Ogni tool registrato con `@mcp.tool()` pubblica uno schema standard:
- `name`: identificatore del tool
- `description`: descrizione testuale generata dalle docstring di Python (usata da Llama per capire quando invocarlo)
- `parameters`: schema dei tipi di input

In [10]:
print("--- 🔧 2. TOOLS ESPOSTI E SCHEMI JSON ---")
tools_schema = get_mcp_tools()

for tool in tools_schema:
    fn = tool["function"]
    print(f"Tool: {fn['name']}")
    print(f"  Descrizione: {fn['description']}")
    print(f"  Parametri Richiesti: {fn['parameters'].get('required', [])}")
    print(f"  Proprietà: {list(fn['parameters']['properties'].keys())}")
    print("-" * 50)

--- 🔧 2. TOOLS ESPOSTI E SCHEMI JSON ---
Tool: get_employee_budget
  Descrizione: Passo 1 obbligatorio: Recupera budget residuo, ruolo e reparto del dipendente aziendale (es. 'E4521').
  Parametri Richiesti: ['employee_id']
  Proprietà: ['employee_id']
--------------------------------------------------
Tool: search_product
  Descrizione: Cerca nel catalogo hardware per trovare SKU, nome esatto e prezzo in Euro.
    Accetta termini di ricerca flessibili (es. 'ThinkPad', 'laptop', 'monitor', '4k', 'dell', 'macbook').
  Parametri Richiesti: ['query']
  Proprietà: ['query']
--------------------------------------------------
Tool: request_manager_approval
  Descrizione: DA USARE SOLO SE l'acquisto è FUORI POLICY (prezzo > budget residuo o prezzo > limite del reparto). NON chiamare se l'acquisto è in policy.
  Parametri Richiesti: ['employee_id', 'sku', 'motivazione']
  Proprietà: ['employee_id', 'sku', 'motivazione']
--------------------------------------------------
Tool: check_order_statu

## 3. Anatomia di una Tool Call (Chiamata e Risposta Diretta)
Proviamo a invocare i tool sul server MCP in esecuzione per visualizzare i payload JSON.

In [11]:
# 1. Recupero budget impiegato E4521 (Mario Rossi)
res_budget = execute_mcp_tool(name="get_employee_budget", args={"employee_id": "E4521"})
print("1. Risposta get_employee_budget:")
pprint(res_budget)

print("\n" + "="*50 + "\n")

# 2. Ricerca laptop nel catalogo
res_catalog = execute_mcp_tool(name="search_product", args={"categoria": "laptop"})
print("2. Risposta search_product:")
pprint(res_catalog)

print("\n" + "="*50 + "\n")

# 3. Gestione errore (ID inesistente)
res_err = execute_mcp_tool(name="get_employee_budget", args={"employee_id": "E_NON_ESISTE"})
print("3. Risposta Errore Gestito:")
pprint(res_err)

1. Risposta get_employee_budget:
{'result': {'data': {'budget_residuo': 1800.0,
                     'nome': 'Mario Rossi',
                     'progetto': 'Project Delta',
                     'reparto': 'Engineering',
                     'ruolo': 'Senior Developer'},
            'status': 'success'}}


2. Risposta search_product:
{'error': {'code': 'TOOL_EXECUTION_ERROR',
           'message': 'search_product() got an unexpected keyword argument '
                      "'categoria'"}}


3. Risposta Errore Gestito:
{'result': {'message': "Dipendente 'E_NON_ESISTE' non trovato.",
            'status': 'error'}}


## 4. Tool Calling End-to-End con Llama 3.2 e Ollama
Costruiamo il ciclo agentico:
1. L'utente invia una richiesta (es: *"Posso acquistare questo laptop?"*).
2. L'agente recupera i dati con i tool informativi (`get_employee_budget`, `search_product`).
3. Valuta le regole:
   - Se il prezzo è entro il budget e la policy del reparto -> **Risponde confermando che è in policy ed approvato**, senza inoltrare richieste.
   - Se il prezzo eccede -> Usa `request_manager_approval` per chiedere deroga.
4. Formula la risposta finale chiara all'utente.

```
Utente ──> Host ──(Prompt + Tools)──> Llama 3.2
             ▲                           │
             │                           ▼ [tool_calls]
             └──(HTTP POST /mcp)◄── Server MCP (porta 8000)
             │
             └──(Risultato Tool)──> Llama 3.2 ──> Risposta Finale all'Utente
```

In [ ]:
def run_agent_turn(user_query: str, system_policy: str, max_iterations: int = 5):
    """
    Loop di esecuzione Agentico:
    - Passa il contesto, la policy e gli schemi dei tool a Ollama
    - Gestisce iterativamente le tool_calls sul server MCP
    """
    system_prompt = (
        "Sei l'Assistente Acquisti Hardware aziendale.\n"
        f"=== POLICY AZIENDALE ===\n{system_policy}\n========================\n\n"
        "REGOLE OPERATIVE:\n"
        "1. Interroga SEMPRE 'get_employee_budget' e 'search_product' prima di rispondere.\n"
        "2. CONFRONTO NUMERICO:\n"
        "   - Se Prezzo <= Budget_Residuo E Prezzo <= Limite_Policy: L'acquisto è IN POLICY ed APPROVATO AUTOMATICAMENTE. NON chiamare 'request_manager_approval'.\n"
        "   - Se Prezzo > Budget_Residuo OPPURE Prezzo > Limite_Policy (es: monitor > 300€ o spesa > budget residuo): L'acquisto è EXTRA-BUDGET/FUORI POLICY. In questo caso DEVI INVOCARE il tool 'request_manager_approval(employee_id, sku, motivazione)' per richiedere autorizzazione al manager.\n"
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"{user_query}\n(Esegui get_employee_budget e search_product per verificare i dati)."}
    ]
    
    tools = get_mcp_tools()
    
    print(f"👤 UTENTE: {user_query}\n")
    
    for step in range(max_iterations):
        print(f"--- [Step {step+1}] Invocazione Modello Ollama ({MODEL_NAME}) ---")
        payload = {
            "model": MODEL_NAME,
            "messages": messages,
            "tools": tools,
            "stream": False,
            "options": {"temperature": 0.0}
        }
        
        resp = requests.post(OLLAMA_URL, json=payload)
        if resp.status_code != 200:
            print("❌ Errore chiamata Ollama:", resp.text)
            return None
            
        msg = resp.json().get("message", {})
        tool_calls = msg.get("tool_calls", [])
        messages.append(msg)
            
        # Se non ci sono tool_calls, abbiamo la risposta finale
        if not tool_calls:
            print("\n🤖 RISPOSTA FINALE AGENTE:")
            print(msg.get("content", ""))
            return msg.get("content", "")
            
        # Esecuzione dei tool richiesti tramite MCP
        for call in tool_calls:
            fn_name = call["function"]["name"]
            fn_args = call["function"]["arguments"]
            print(f"👉 Il modello ha richiesto il TOOL MCP: '{fn_name}'")
            print(f"   Argomenti passati: {json.dumps(fn_args, ensure_ascii=False)}")
            
            # Chiamata al Server MCP
            tool_output = execute_mcp_tool(fn_name, fn_args)
            print(f"   Risultato dal Server MCP: {json.dumps(tool_output, ensure_ascii=False)}")
            
            messages.append({
                "role": "tool",
                "name": fn_name,
                "content": json.dumps(tool_output, ensure_ascii=False)
            })
        print()
        
    print("⚠️ Raggiunto limite iterazioni.")
    return None

## 5. Test dei Casi d'Uso Chiave

### Caso 1: Mario Rossi (E4521) - Richiesta in Policy
Mario Rossi è Senior Developer (Engineering), limite 1800€, budget residuo 1800€, e richiede il laptop ThinkPad da 1650€.
**Comportamento atteso**: L'agente verifica budget (1800€) e prezzo (1650€), conferma che l'acquisto è in policy ed approvato automaticamente (senza chiamare il manager).

In [13]:
policy_res = read_mcp_resource("policy://acquisti-hardware")
query_mario = "Ciao, sono Mario Rossi (badge E4521). Vorrei sapere se posso ordinare il laptop ThinkPad P16s per lavorare al Project Delta. Rientra nella policy?"
run_agent_turn(query_mario, policy_res)

👤 UTENTE: Ciao, sono Mario Rossi (badge E4521). Vorrei sapere se posso ordinare il laptop ThinkPad P16s per lavorare al Project Delta. Rientra nella policy?

--- [Step 1] Invocazione Modello Ollama (llama3.2) ---
👉 Il modello ha richiesto il TOOL MCP: 'get_employee_budget'
   Argomenti passati: {"employee_id": "E4521"}
   Risultato dal Server MCP: {"result": {"status": "success", "data": {"nome": "Mario Rossi", "ruolo": "Senior Developer", "reparto": "Engineering", "budget_residuo": 1800.0, "progetto": "Project Delta"}}}
👉 Il modello ha richiesto il TOOL MCP: 'search_product'
   Argomenti passati: {"query": "ThinkPad P16s"}
   Risultato dal Server MCP: {"result": {"status": "success", "trovati": 1, "prodotti": [{"sku": "LAP-DEV-01", "nome": "ThinkPad P16s / MacBook Pro", "categoria": "laptop", "prezzo": 1650.0}]}}

--- [Step 2] Invocazione Modello Ollama (llama3.2) ---

🤖 RISPOSTA FINALE AGENTE:
Ciao Mario Rossi (E4521), grazie alla tua richiesta, ho verificato i dati e posso informart

"Ciao Mario Rossi (E4521), grazie alla tua richiesta, ho verificato i dati e posso informarti che il laptop ThinkPad P16s rientra nella policy aziendale per il tuo reparto (Engineering) e per il tuo budget residuo (1800€).\n\nPoiché il prezzo del laptop ThinkPad P16s (1650€) è inferiore alla tua limitazione di spesa (1800€), l'acquisto è automaticamente approvato. Non è necessario richiedere approvazione al manager.\n\nPotrai therefore ordinare il laptop ThinkPad P16s per lavorare al Project Delta."

### Caso 2: Giulia Verdi (E1082) - Fuori Budget (Approvazione Manager)
Giulia Verdi è nel Marketing (limite 1000€), budget residuo 450€, e richiede un Monitor 4K da 520€.
**Comportamento atteso**: Il costo (520€) supera il budget residuo (450€) e il limite monitor (300€). L'agente inoltra la richiesta al manager tramite `request_manager_approval` e restituisce il ticket.

In [14]:
query_giulia = "Salve! Sono Giulia Verdi (badge E1082) del team Marketing. Ho bisogno del Monitor 4K per la campagna. Posso procedere all'acquisto?"
run_agent_turn(query_giulia, policy_res)

👤 UTENTE: Salve! Sono Giulia Verdi (badge E1082) del team Marketing. Ho bisogno del Monitor 4K per la campagna. Posso procedere all'acquisto?

--- [Step 1] Invocazione Modello Ollama (llama3.2) ---
👉 Il modello ha richiesto il TOOL MCP: 'get_employee_budget'
   Argomenti passati: {"employee_id": "E1082"}
   Risultato dal Server MCP: {"result": {"status": "success", "data": {"nome": "Giulia Verdi", "ruolo": "Content Specialist", "reparto": "Marketing", "budget_residuo": 450.0, "progetto": "Brand Campaign Q2"}}}
👉 Il modello ha richiesto il TOOL MCP: 'search_product'
   Argomenti passati: {"query": "Monitor 4K"}
   Risultato dal Server MCP: {"result": {"status": "success", "trovati": 1, "prodotti": [{"sku": "MON-4K-01", "nome": "Monitor Dell 4K 27''", "categoria": "monitor", "prezzo": 520.0}]}}

--- [Step 2] Invocazione Modello Ollama (llama3.2) ---

🤖 RISPOSTA FINALE AGENTE:
Ciao Giulia Verdi,

Per quanto riguarda l'acquisto del Monitor 4K, abbiamo trovato un prodotto che corrisponde al

'Ciao Giulia Verdi,\n\nPer quanto riguarda l\'acquisto del Monitor 4K, abbiamo trovato un prodotto che corrisponde alla tua richiesta. Il Monitor Dell 4K 27\'\' ha un prezzo di 520€, che supera la tua disponibilità di budget residuo di 450€.\n\nIn base alle nostre regole operative, poiché il prezzo del prodotto supera la tua disponibilità di budget residuo, dobbiamo richiedere l\'approvazione del manager per procedere con l\'acquisto.\n\nTi invio un messaggio di richiesta di approvazione al manager:\n\n"Richiesta di approvazione per acquisto Monitor Dell 4K 27\'\' (SKU: MON-4K-01) per la campagna di marketing. Il prezzo del prodotto è di 520€, che supera la disponibilità di budget residuo di 450€. La motivazione per l\'acquisto è la necessità di un monitor di alta qualità per la campagna di marketing."\n\nSpero che questo ti sia stato utile. Se hai altre domande o richieste, non esitare a chiedere.'

## 6. 💡 Esercizio Pratico per lo Studente
Aggiungi un nuovo tool `@mcp.tool()` all'interno del server in [`simple_mcp_server.py`](simple_mcp_server.py):

**Specifiche:**
```python
@mcp.tool()
def check_order_status(order_id: str) -> dict:
    """Verifica lo stato di spedizione di un ordine esistente dato il suo ID (es. ORD-123)."""
    if order_id.startswith("ORD-"):
        return {"status": "in_transit", "estimated_delivery": "2 giorni lavorativi"}
    return {"status": "not_found", "message": "ID ordine non valido"}
```
Riavvia il server (`python simple_mcp_server.py`) e testa la query: *"Qual è lo stato dell'ordine ORD-9876?"*

In [15]:
execute_mcp_tool(
    name="check_order_status",
    args={"order_id": "ORD-9876"}
)

{'result': {'status': 'in_transit',
  'estimated_delivery': '2 giorni lavorativi'}}